In [1]:
import pandas as pd
import numpy as np
import nfl_data_py as nfl
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [2]:
# seasons = range(2006, 2024)     #Passing stats started in 2006
pbp_py = nfl.import_pbp_data([2023])
pbp_py_2016 = nfl.import_pbp_data([2016])

2023 done.
Downcasting floats.
2016 done.
Downcasting floats.


In [3]:
pbp_py[:10].to_csv('small_data.csv')

From defensive perspective, predicting what the offense is going to run based on situational context (time reamining, downs, distance) and offensive formation

In [11]:
game_situation = ["yardline_100", "quarter_seconds_remaining", "half_seconds_remaining", "game_seconds_remaining", "drive", "qtr", "down", "ydstogo", "score_differential", "xpass"] 
offensive = ["offense_formation", "offense_personnel"]
defensive = ["defenders_in_box", "defense_personnel", "number_of_pass_rushers", "defense_man_zone_type", "defense_coverage_type"]
play_type = ["play_type", "run_location", "run_gap", "pass_location", "pass_length"]

df = pbp_py[game_situation + offensive + defensive + play_type]
df[:5]

,yardline_100,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,drive,qtr,down,ydstogo,score_differential,xpass,...,defenders_in_box,defense_personnel,number_of_pass_rushers,defense_man_zone_type,defense_coverage_type,play_type,run_location,run_gap,pass_location,pass_length
0,NaN,900.0,1800.0,3600.0,NaN,1.0,NaN,0.0,NaN,NaN,...,NaN,None,NaN,None,None,None,None,None,None,None
1,35.0,900.0,1800.0,3600.0,1.0,1.0,NaN,0.0,0.0,NaN,...,NaN,None,NaN,None,None,kickoff,None,None,None,None
2,75.0,900.0,1800.0,3600.0,1.0,1.0,1.0,10.0,0.0,0.515058,...,7.0,"3 DL, 4 LB, 4 DB",NaN,None,None,run,right,tackle,None,None
3,72.0,870.0,1770.0,3570.0,1.0,1.0,2.0,7.0,0.0,0.661106,...,6.0,"2 DL, 4 LB, 5 DB",4.0,ZONE_COVERAGE,COVER_3,pass,None,None,right,short
4,66.0,835.0,1735.0,3535.0,1.0,1.0,3.0,1.0,0.0,0.196065,...,6.0,"2 DL, 4 LB, 5 DB",NaN,None,None,run,left,guard,None,None


In [12]:
df = df.dropna(subset=["yardline_100", "play_type"]).reset_index(drop=True)
df = df.loc[df['play_type'].isin(['run', 'pass'])]
df[:5]
len(df)

35600

In [13]:
df_combined = df.copy()

def combine_columns(row):
    if row['play_type'] == 'run':
        return f"{row['play_type']}_{row['run_location']}_{row['run_gap']}"
    elif row['play_type'] == 'pass':
        return f"{row['play_type']}_{row['pass_location']}_{row['pass_length']}"
    else:
        return None  # or any default value

df_combined['combined'] = df_combined.apply(combine_columns, axis=1)


In [18]:
# df_combined = df_combined.drop(["play_type", "run_location", "run_gap", "pass_location", "pass")
df_combined = df_combined[~df_combined['combined'].isin(['run_right_None', 'pass_None_None', 'run_None_None'])]

In [19]:
df_combined.groupby("combined").count()

,yardline_100,quarter_seconds_remaining,half_seconds_remaining,game_seconds_remaining,drive,qtr,down,ydstogo,score_differential,xpass,...,defenders_in_box,defense_personnel,number_of_pass_rushers,defense_man_zone_type,defense_coverage_type,play_type,run_location,run_gap,pass_location,pass_length
combined,,,,,,,,,,,,,,,,,,,,,
pass_left_deep,1417,1417,1417,1417,1417,1417,1417,1417,1417,1417,...,1241,1241,1241,1238,1238,1417,0,0,1417,1417
pass_left_short,5994,5994,5994,5994,5994,5994,5994,5994,5994,5994,...,5210,5213,5213,5206,5206,5994,0,0,5994,5994
pass_middle_deep,621,621,621,621,621,621,621,621,621,621,...,541,541,541,541,541,621,0,0,621,621
pass_middle_short,3144,3144,3144,3144,3144,3144,3144,3144,3144,3144,...,2759,2759,2759,2752,2752,3144,0,0,3144,3144
pass_right_deep,1448,1448,1448,1448,1448,1448,1448,1448,1448,1448,...,1286,1287,1287,1283,1283,1448,0,0,1448,1448
pass_right_short,6560,6560,6560,6560,6560,6560,6560,6560,6560,6560,...,5714,5717,5717,5710,5710,6560,0,0,6560,6560
run_left_end,2048,2048,2048,2048,2048,2048,2048,2048,2048,2048,...,1750,1751,264,0,0,2048,2048,2048,0,0
run_left_guard,1756,1756,1756,1756,1756,1756,1756,1756,1756,1756,...,1509,1509,26,0,0,1756,1756,1756,0,0
run_left_tackle,1705,1705,1705,1705,1705,1705,1705,1705,1705,1705,...,1495,1496,44,0,0,1705,1705,1705,0,0


In [20]:
X = pd.get_dummies(df_combined[game_situation + offensive + defensive])
y = df_combined["combined"]

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_))

                   precision    recall  f1-score   support

   pass_left_deep       0.14      0.02      0.04       283
  pass_left_short       0.30      0.40      0.35      1199
 pass_middle_deep       0.08      0.01      0.01       124
pass_middle_short       0.21      0.07      0.11       629
  pass_right_deep       0.08      0.01      0.02       290
 pass_right_short       0.34      0.53      0.42      1312
     run_left_end       0.17      0.14      0.15       410
   run_left_guard       0.10      0.06      0.08       351
  run_left_tackle       0.17      0.11      0.13       341
  run_middle_None       0.30      0.55      0.39       781
    run_right_end       0.21      0.17      0.19       390
  run_right_guard       0.15      0.09      0.11       370
 run_right_tackle       0.13      0.06      0.08       302

         accuracy                           0.28      6782
        macro avg       0.18      0.17      0.16      6782
     weighted avg       0.24      0.28      0.24      

<h2>GridSearchCV</h2>

In [22]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'max_features': ['sqrt', 'log2', None]
}

clf = DecisionTreeClassifier(random_state=42)

grid_search = GridSearchCV(
    estimator=clf,
    param_grid=param_grid,
    cv=5,                     # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,                # Use all CPU cores
    verbose=1
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters: {'criterion': 'entropy', 'max_depth': 5, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Score: 0.29730131316014846
Test Accuracy: 0.298879386611619
